# Cell Translator — morphology category enrichment, top-20 effects and integrated gradients

**Companion analysis to** `Morph_perturbation_corrected_QC_70_15_15_v3.ipynb`.

Uses **saved original-scale test morphology** for the three drug perturbations and both translation strategies. No AE or CycleGAN retraining. The definitions of feature categories, classical Cohen’s d, top-decile category enrichment, and source-feature IG follow the original `Morph_perturbation.ipynb` (especially the original feature-family/effect and IG analysis cells), with differences explicitly noted below.

**Two distinct questions:**
- **Drug effects:** which morphological features/families change relative to DMSO in the measured data, and do translations recover those changes?
- **Model reliance:** which **source morphology features** influence the predicted **target latent-state difference from a reference source**? IG is a model attribution, **not** a direct measure of drug response or causal biological importance.

**Enrichment terminology:** `enriched/depleted` = **over/underrepresented** within a selected top-effect/IG feature list; `positive/negative` = **direction of the signed Cohen's d**. These are not interchangeable.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom, pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

BASE_DIR = Path(os.environ.get("CELL_TRANSLATOR_DATA_DIR", str(Path.cwd() / "data"))).expanduser().resolve()
RUN_ROOT = Path(os.environ.get('CELL_TRANSLATOR_RUN_DIR', str(Path.cwd() / 'outputs' / 'rerun_70_15_15_qc_v3'))).expanduser().resolve()
AE_DIR = RUN_ROOT / 'autoencoders'
DEV_DIR = RUN_ROOT / 'cyclegan_ipsc_astrocyte_drugs'
CTRL_DIR = RUN_ROOT / 'cyclegan_control_to_perturbation'
EVAL_DIR = RUN_ROOT / 'evaluation'
OUT_DIR = EVAL_DIR / 'category_effect_ig'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DRUGS = ['Latrunculin_B', 'GW3965', 'Clozapine']
GROUP_ORDER = ['Texture', 'Intensity', 'Granularity', 'RadialDistribution', 'AreaShape', 'Correlation']
TOP_N = 20
TOP_QUANTILE = 0.90
IG_N_CELLS = 128     # representative source test cells; consider 256 for sensitivity analysis
IG_STEPS = 32
IG_BATCH_SIZE = 16
SEED = 42

print('Reading results from:', RUN_ROOT)
print('Writing analysis to:', OUT_DIR)
assert AE_DIR.exists() and DEV_DIR.exists() and CTRL_DIR.exists(), 'Check RUN_ROOT and completed training outputs'

## 1. Load original-scale measured and predicted test profiles

Read saved **per-condition test data**, inverse-standardized with its own training-fitted scaler. For each drug, align by **feature name**; never match arrays by column position or row number. Measured and translated populations are unpaired. Each strategy has a distinct reference: developmental uses predicted DMSO astrocytes; control-to-drug uses measured DMSO astrocytes.

In [ ]:
def get_real_test(condition):
    prefix = AE_DIR / condition
    stats = pd.read_csv(str(prefix) + '_scaler_stats.csv')
    x = np.load(str(prefix) + '_X_test_scaled.npy')
    assert x.shape[1] == len(stats), f'{condition}: scaler columns mismatch'
    x = x.astype(float) * stats['scaler_scale'].to_numpy()[None, :] + stats['scaler_mean'].to_numpy()[None, :]
    return pd.DataFrame(x, columns=stats['feature'].tolist())

def load_pred_dev(drug):
    p = DEV_DIR / drug
    x = np.load(p / 'translated_astro_morphology_original.npy')
    features = pd.read_csv(p / 'translated_astro_feature_names.csv')['feature'].tolist()
    assert x.shape[1] == len(features), f'{drug}: developmental feature mismatch'
    return pd.DataFrame(x, columns=features)

def load_pred_ctrl(cell_type, drug):
    p = CTRL_DIR / f'{cell_type}_DMSO_to_{drug}'
    x = np.load(p / 'X_control_to_drug_original.npy')
    features = pd.read_csv(p / 'control_to_drug_feature_names.csv')['feature'].tolist()
    assert x.shape[1] == len(features), f'{cell_type} {drug}: control feature mismatch'
    return pd.DataFrame(x, columns=features)

def morphology_family(feature):
    parts = str(feature).strip().split('_')
    for group in GROUP_ORDER:
        if (len(parts)>1 and parts[1]==group) or str(feature).startswith(group+'_') or ('_'+group+'_') in str(feature):
            return group
    return 'Unknown'

def classical_cohens_d(control, drug, tol=1e-8):
    control, drug = np.asarray(control, float), np.asarray(drug, float)
    if control.shape[0] < 2 or drug.shape[0] < 2: raise ValueError('At least two cells are required per state')
    n0, n1 = control.shape[0], drug.shape[0]
    m0, m1 = np.nanmean(control, axis=0), np.nanmean(drug, axis=0)
    v0, v1 = np.nanvar(control, axis=0, ddof=1), np.nanvar(drug, axis=0, ddof=1)
    sd = np.sqrt(((n0-1)*v0 + (n1-1)*v1)/(n0+n1-2))
    d = np.full(len(m0), np.nan)
    ok = np.isfinite(sd)&(sd>tol)&np.isfinite(m0)&np.isfinite(m1)
    d[ok] = (m1[ok]-m0[ok])/sd[ok]
    return d, sd, m1-m0

def align_four(r0, r1, p0, p1):
    common = [f for f in r1.columns if f in r0.columns and f in p0.columns and f in p1.columns]
    if len(common) < 3: raise ValueError('Not enough common named morphology features')
    return common, [x[common].to_numpy(dtype=np.float64) for x in (r0,r1,p0,p1)]

def make_effect_table(strategy, drug):
    real0 = get_real_test('Astrocytes_DMSO')
    real1 = get_real_test('Astrocytes_'+drug)
    if strategy == 'developmental':
        pred0, pred1 = load_pred_dev('DMSO'), load_pred_dev(drug)
    elif strategy == 'control_to_drug':
        pred0, pred1 = real0, load_pred_ctrl('Astrocytes',drug)
    else: raise ValueError(strategy)
    features, (R0,R1,P0,P1) = align_four(real0,real1,pred0,pred1)
    dm, sd_real, diffm = classical_cohens_d(R0,R1)
    dp, sd_pred, diffp = classical_cohens_d(P0,P1)
    # Comparability check: both mean changes / SAME measured pooled SD.
    dpsame = np.divide(diffp, sd_real, out=np.full_like(diffp,np.nan),where=np.isfinite(sd_real)&(sd_real>1e-8))
    df = pd.DataFrame(dict(feature=features, family=[morphology_family(f) for f in features],
        d_measured=dm, d_predicted_own_pooled_sd=dp, d_predicted_measured_pooled_sd=dpsame,
        abs_d_measured=np.abs(dm), abs_d_predicted=np.abs(dp),
        abs_d_error=np.abs(dm-dpsame), sign_match=np.sign(dm)==np.sign(dpsame)))
    df['strategy']=strategy;df['drug']=drug
    return df.replace([np.inf,-np.inf],np.nan)

effect_tables={}
for strategy in ['developmental','control_to_drug']:
    for drug in DRUGS:
        df=make_effect_table(strategy,drug)
        label=f'{strategy}__{drug}'
        df.to_csv(OUT_DIR / f'{label}_cohens_d_features.csv', index=False)
        effect_tables[label]=df
        print(label,len(df),'common features,',df['d_measured'].notna().sum(),'valid measured d')

## 2. Top 20 measured and predicted Cohen's d features

Top 20 is ranked by **absolute Cohen's d**, but bars preserve the positive or negative sign. Two predicted columns are retained:
- `d_predicted_own_pooled_sd`: classical Cohen's d for the predicted populations themselves (for independent ranked lists).
- `d_predicted_measured_pooled_sd`: predicted mean shift divided by the **measured pooled SD**, so magnitude error can be compared fairly with measured drug effects (same denominator as your earlier drug-effect recovery tables).

The top 20 measured and predicted lists may differ. This code saves both and visualizes each separately. It also compares predictions on the **same measured top 20** to avoid selection bias from choosing features independently on each side.

In [ ]:
def plot_top_20(df, label, top_n=TOP_N):
    valid=df.dropna(subset=['d_measured','d_predicted_own_pooled_sd']).copy()
    defs=[('Measured', 'd_measured'),('Predicted','d_predicted_own_pooled_sd')]
    fig, axes=plt.subplots(1,2,figsize=(14,8),sharex=False)
    for ax,(title,col) in zip(axes,defs):
        top=valid.loc[valid[col].abs().nlargest(min(top_n,len(valid))).index].copy()
        top=top.iloc[::-1]
        ax.barh(range(len(top)),top[col],color=['#b85c6c' if d<0 else '#427e99' for d in top[col]])
        ax.set_yticks(range(len(top)),top['feature'],fontsize=7)
        ax.axvline(0,color='gray',linewidth=0.7)
        ax.set_xlabel('Signed Cohen’s d')
        ax.set_title(title+' — top absolute effects')
        top.assign(rank=np.arange(len(top),0,-1)).to_csv(OUT_DIR/f'{label}_top{top_n}_{title.lower()}.csv',index=False)
    fig.suptitle(label.replace('__',' — '))
    fig.tight_layout()
    fig.savefig(OUT_DIR/f'{label}_top{top_n}_cohens_d.png',dpi=240,bbox_inches='tight')
    plt.close(fig)

    measured_top=set(valid.nlargest(top_n,'abs_d_measured')['feature'])
    predicted_top=set(valid.nlargest(top_n,'abs_d_predicted')['feature'])
    overlap=len(measured_top&predicted_top)
    return {'comparison':label,'top_n':top_n,'n_common_features':len(valid),'overlap':overlap,
            'jaccard':overlap/len(measured_top|predicted_top),
            'measured_top20_sign_accuracy':float(valid.set_index('feature').loc[list(measured_top),'sign_match'].mean())}

summary=[plot_top_20(tab,label) for label,tab in effect_tables.items()]
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(OUT_DIR/'top20_overlap_summary.csv',index=False)

## 3. Category enrichment / depletion

This extends your original `enrichment_raw()` analysis, which used a top-decile threshold and observed/expected representation. We now compute explicit **overrepresentation and underrepresentation** one-sided hypergeometric p-values (equivalent to fixed-margin Fisher tests), with BH-FDR across all category tests in this analysis.

Three feature selections are made for each drug and translation strategy:
1. `measured_abs`: top 10% largest measured absolute Cohen's d;
2. `predicted_abs`: top 10% largest predicted absolute Cohen's d;
3. `measured_positive` and `measured_negative`: top 10% of *signed* measured effects for direction-specific feature-family representation. These are **different** from family-level mean effects.

`obs_exp > 1` means the category is **enriched among selected features**, `obs_exp < 1` means **depleted**. A family can have predominantly negative effects and still be enriched.

In [ ]:
def category_enrichment(df, score_column, label, q=TOP_QUANTILE):
    w=df[['feature','family',score_column]].dropna().copy()
    w=w[w.family.isin(GROUP_ORDER)].copy()  # background excludes Unknown for an interpretable family universe
    if w.empty: raise ValueError(f'{label}: no valid features for enrichment')
    cutoff=w[score_column].quantile(q)
    high=w[score_column]>=cutoff
    N=len(w); K=int(high.sum())
    result=[]
    for family in GROUP_ORDER:
        group=w.family==family
        n=int(group.sum())
        if n==0:continue
        k=int((group & high).sum())
        expected=K*n/N
        result.append({'analysis':label,'family':family,'N_background':N,'K_selected':K,
                       'n_family':n,'k_selected_family':k,'expected_selected':expected,
                       'obs_exp':k/expected if expected else np.nan,
                       'p_enriched':float(hypergeom.sf(k-1,N,n,K)),
                       'p_depleted':float(hypergeom.cdf(k,N,n,K)),
                       'threshold':cutoff})
    return pd.DataFrame(result)

enrich_parts=[]
for label,tab in effect_tables.items():
    for score,desc in [('abs_d_measured','measured_abs'),('abs_d_predicted','predicted_abs'),
                       ('d_measured','measured_positive')]:
        enrich_parts.append(category_enrichment(tab,score,label+'__'+desc))
    tmp=tab.copy();tmp['negative_effect']=-tmp['d_measured']
    enrich_parts.append(category_enrichment(tmp,'negative_effect',label+'__measured_negative'))

enrichment=pd.concat(enrich_parts,ignore_index=True)
# Adjust across all over- and underrepresentation tests (and all drugs/strategies/selections).
all_p=np.r_[enrichment.p_enriched.to_numpy(),enrichment.p_depleted.to_numpy()]
all_q=multipletests(all_p,method='fdr_bh')[1]
enrichment['q_enriched']=all_q[:len(enrichment)]
enrichment['q_depleted']=all_q[len(enrichment):]
enrichment['direction_of_representation']=np.where(enrichment.obs_exp>=1,'enriched','depleted')
enrichment['q_direction']=np.where(enrichment.obs_exp>=1,enrichment.q_enriched,enrichment.q_depleted)
enrichment.to_csv(OUT_DIR/'category_enrichment_measured_predicted.csv',index=False)
display(enrichment.query('q_direction < 0.05').sort_values('q_direction').head(30))

In [ ]:
def enrichment_plot(df, analysis, outfile):
    g=df[df.analysis==analysis].set_index('family').reindex(GROUP_ORDER)
    fig,ax=plt.subplots(figsize=(9,4.4))
    vals=g.obs_exp
    colors=['#427e99' if (np.isfinite(v) and v>=1) else '#b85c6c' for v in vals]
    ax.barh(GROUP_ORDER,vals,color=colors)
    ax.axvline(1,color='black',linestyle='--',linewidth=1)
    for y,(v,qval) in enumerate(zip(vals,g.q_direction)):
        if np.isfinite(v) and np.isfinite(qval) and qval<0.05:
            ax.text(max(0,v)+0.02,y,'*',va='center')
    ax.set_xlabel('Observed / expected among selected features')
    ax.set_title(analysis.replace('__',' | '))
    ax.set_xlim(left=0)
    fig.tight_layout()
    fig.savefig(outfile,dpi=230,bbox_inches='tight');plt.close(fig)

for label in effect_tables:
    for desc in ['measured_abs','predicted_abs','measured_positive','measured_negative']:
        analysis=f'{label}__{desc}'
        enrichment_plot(enrichment,analysis,OUT_DIR/f'{analysis}_enrichment.png')

## 4. Feature-family effect summary (signed and absolute)

Enrichment counts tell you whether categories are overrepresented among top features. This supplementary table directly characterizes the **effect direction and magnitude** by family (averaged feature-wise, not across individual cells). Avoid treating these highly correlated CellProfiler features as independent biological replicates.

In [ ]:
family_rows=[]
for label,df in effect_tables.items():
    for family,g in df.groupby('family'):
        g=g.dropna(subset=['d_measured','d_predicted_measured_pooled_sd'])
        if len(g)==0:continue
        family_rows.append({'comparison':label,'family':family,'n_features':len(g),
            'mean_signed_d_measured':g.d_measured.mean(),
            'mean_signed_d_predicted':g.d_predicted_measured_pooled_sd.mean(),
            'mean_abs_d_measured':g.d_measured.abs().mean(),
            'mean_abs_d_predicted':g.d_predicted_measured_pooled_sd.abs().mean(),
            'sign_concordance':g.sign_match.mean(),
            'mean_abs_error':g.abs_d_error.mean()})
family_summary=pd.DataFrame(family_rows)
family_summary.to_csv(OUT_DIR/'morphology_family_effect_summary.csv',index=False)
display(family_summary)

## 5. Integrated gradients of source morphology features (both translation strategies)

**Same attribution objective as the old notebook:** IG of \(rac12\|G(E(x))-G(E(b))\|_2^2\), where \(x\) is a standardized **source** morphology profile, \(b\) is the training-population mean source profile, \(E\) is the frozen source AE encoder, and \(G\) is the saved forward CycleGAN generator. The encoder's output is standardized with saved train-derived latent mean/SD before entering \(G\). The output is the **target standardized latent state**.

We transfer the trained TensorFlow generator weights into an equivalent PyTorch module to backpropagate through the PyTorch source AE encoder. This follows your original notebook's `TorchForwardGenerator`, TensorFlow→PyTorch weight copying, `FullTranslationForIG`, and trapezoidal `integrated_gradients_translation` logic.

**Critical check:** A TensorFlow-vs-PyTorch forward-output numerical agreement test must pass before calculating IG. Unlike effect recovery, IG does not use real target cells. The output rows contain **source-feature** categories.

In [ ]:
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow.keras import layers,models

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def tf_core_model(dim, name):
    inp=layers.Input(shape=(dim,))
    x=inp
    for units in [512,256,256]:
        x=layers.Dense(units,activation='relu')(x)
        x=layers.LayerNormalization()(x)
    x=layers.Dense(dim)(x)
    return models.Model(inp,x,name=name)

def build_tf_saved_generator(dim, strategy):
    name='G_SA_core' if strategy=='developmental' else 'G_control_to_drug_core'
    core=tf_core_model(dim,name)
    inp=layers.Input(shape=(dim,))
    x=core(inp)
    y=layers.Dense(dim,use_bias=True,name='linear_head')(x)
    return models.Model(inp,y,name='G_SA' if strategy=='developmental' else 'G_control_to_drug')

class TorchForwardGenerator(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.fc1=nn.Linear(dim,512);self.ln1=nn.LayerNorm(512,eps=1e-3)
        self.fc2=nn.Linear(512,256);self.ln2=nn.LayerNorm(256,eps=1e-3)
        self.fc3=nn.Linear(256,256);self.ln3=nn.LayerNorm(256,eps=1e-3)
        self.fc4=nn.Linear(256,dim);self.head=nn.Linear(dim,dim)
    def forward(self,x):
        x=self.ln1(torch.relu(self.fc1(x)))
        x=self.ln2(torch.relu(self.fc2(x)))
        x=self.ln3(torch.relu(self.fc3(x)))
        return self.head(self.fc4(x))

def transfer_generator(tf_gen,dim):
    pt=TorchForwardGenerator(dim)
    core=tf_gen.get_layer('G_SA_core' if tf_gen.name=='G_SA' else 'G_control_to_drug_core')
    dense=[v for v in core.layers if isinstance(v,layers.Dense)]
    layernorm=[v for v in core.layers if isinstance(v,layers.LayerNormalization)]
    assert len(dense)==4 and len(layernorm)==3, 'Unexpected TensorFlow generator architecture'
    def copy_dense(t,p):
        w,b=t.get_weights()
        with torch.no_grad():
            p.weight.copy_(torch.from_numpy(w.T));p.bias.copy_(torch.from_numpy(b))
    for t,p in zip(dense,[pt.fc1,pt.fc2,pt.fc3,pt.fc4]):copy_dense(t,p)
    for t,p in zip(layernorm,[pt.ln1,pt.ln2,pt.ln3]):
        gamma,beta=t.get_weights()
        with torch.no_grad():
            p.weight.copy_(torch.from_numpy(gamma));p.bias.copy_(torch.from_numpy(beta))
    copy_dense(tf_gen.get_layer('linear_head'),pt.head)
    return pt.eval().to(DEVICE)

class FullTranslationForIG(nn.Module):
    def __init__(self,encoder,generator,src_mean,src_std):
        super().__init__()
        self.encoder=encoder
        self.generator=generator
        self.register_buffer('src_mean',torch.as_tensor(src_mean,dtype=torch.float32))
        self.register_buffer('src_std',torch.as_tensor(src_std,dtype=torch.float32))
    def forward(self,x):
        z=self.encoder(x)
        return self.generator((z-self.src_mean)/self.src_std)

In [ ]:
# This loader assumes the saved AE config and MorphologyAutoencoder class
# from the corrected rerun. If executing standalone in a new kernel, run ONLY
# its architecture definition (the cell defining MorphologyAutoencoder), not training.
assert 'MorphologyAutoencoder' in globals(), ('Define MorphologyAutoencoder from the rerun notebook first. '
                                                'Do not re-run any AE training.')

def load_source_ae_and_data(condition):
    prefix=AE_DIR/condition
    ckpt=torch.load(str(prefix)+'_ae_checkpoint.pt',map_location='cpu',weights_only=False)
    stats=pd.read_csv(str(prefix)+'_scaler_stats.csv')
    dim=len(stats)
    latent_dim=int(ckpt['config']['latent_dim'])
    model=MorphologyAutoencoder(input_dim=dim,latent_dim=latent_dim).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict']);model.eval()
    for p in model.parameters():p.requires_grad_(False)
    Xtr=np.load(str(prefix)+'_X_train_scaled.npy').astype(np.float32)
    Xte=np.load(str(prefix)+'_X_test_scaled.npy').astype(np.float32)
    assert Xtr.shape[1]==Xte.shape[1]==dim
    return model,Xtr,Xte,stats['feature'].tolist(),latent_dim

def load_translation_ig(cell_type,drug,strategy):
    src_condition = f'iPSC_{drug}' if strategy=='developmental' else f'{cell_type}_DMSO'
    ae,Xtr,Xte,features,dim=load_source_ae_and_data(src_condition)
    if strategy=='developmental':
        folder=DEV_DIR/drug
        weights=folder/'G_SA.weights.h5'
        latent=np.load(folder/'latent_scaler_stats.npz')
        mean,std=latent['mean_src'],latent['std_src']
    else:
        folder=CTRL_DIR/f'{cell_type}_DMSO_to_{drug}'
        weights=folder/'G_control_to_drug.weights.h5'
        latent=np.load(folder/'latent_standardization_stats.npz')
        mean,std=latent['src_mean'],latent['src_std']
    tf_gen=build_tf_saved_generator(dim,strategy)
    tf_gen.load_weights(str(weights))
    pt_gen=transfer_generator(tf_gen,dim)
    # Compare forward generator numerical predictions for fixed inputs BEFORE any IG.
    rng=np.random.default_rng(SEED)
    probe=rng.normal(size=(8,dim)).astype(np.float32)
    a=tf_gen(probe,training=False).numpy()
    with torch.no_grad():b=pt_gen(torch.as_tensor(probe,device=DEVICE)).cpu().numpy()
    err=np.max(np.abs(a-b))
    if not np.allclose(a,b,rtol=2e-4,atol=2e-4):
        raise RuntimeError(f'{strategy}/{drug}: TF↔Torch generator mismatch (max error={err:.5g})')
    print(f'{strategy}/{cell_type}/{drug}: generator check max_abs_error={err:.3g}')
    full=FullTranslationForIG(ae.encoder,pt_gen,mean,std).to(DEVICE).eval()
    for p in full.parameters():p.requires_grad_(False)
    return full,Xtr,Xte,features

In [ ]:
def integrated_gradients_translation(model,X,baseline,device=DEVICE,n_steps=IG_STEPS,batch_size=IG_BATCH_SIZE):
    """Trapezoidal IG, using the original notebook's scalar output objective."""
    X=np.asarray(X,dtype=np.float32)
    b=torch.as_tensor(baseline,dtype=torch.float32,device=device).reshape(1,-1)
    model.eval()
    with torch.no_grad():y_base=model(b)
    ig_parts=[]
    for lo in range(0,len(X),batch_size):
        xb=torch.as_tensor(X[lo:lo+batch_size],device=device)
        delta=xb-b
        total=torch.zeros_like(xb)
        for k in range(n_steps+1):
            alpha=k/n_steps
            interp=(b+alpha*delta).detach().requires_grad_(True)
            output=model(interp)
            score=0.5*((output-y_base)**2).sum(dim=1)
            gradient=torch.autograd.grad(score.sum(),interp)[0]
            total=total+(0.5 if k in (0,n_steps) else 1.0)*gradient
        ig_parts.append((delta*total/n_steps).detach().cpu().numpy())
    return np.concatenate(ig_parts,axis=0)

# Sample a reproducible subset of the SOURCE test set and save the selected indices.
def compute_one_ig(strategy,cell_type,drug):
    model,Xtr,Xte,features=load_translation_ig(cell_type,drug,strategy)
    rng=np.random.default_rng(SEED)
    selected=np.sort(rng.choice(len(Xte),size=min(IG_N_CELLS,len(Xte)),replace=False))
    baseline=Xtr.mean(axis=0)   # source training-population mean in source standardized morphology space
    IG=integrated_gradients_translation(model,Xte[selected],baseline)
    assert IG.shape==(len(selected),len(features)) and np.isfinite(IG).all()
    label=f'{strategy}__{cell_type}__{drug}'
    np.save(OUT_DIR/f'{label}_ig_source_test_indices.npy',selected)
    table=pd.DataFrame({'feature':features,'family':[morphology_family(f) for f in features],
        'IG_mean_abs':np.mean(np.abs(IG),axis=0),'IG_mean_signed':np.mean(IG,axis=0),
        'n_ig_test_cells':len(selected)})
    table=table.sort_values('IG_mean_abs',ascending=False)
    table.to_csv(OUT_DIR/f'{label}_source_feature_IG.csv',index=False)
    print(label,':',IG.shape,'source IG matrix')
    return label,table

ig_tables={}
for drug in DRUGS:
    # Each developmental drug is its own perturbed iPSC -> astrocyte model.
    key,tab=compute_one_ig('developmental','iPSC',drug)
    ig_tables[key]=tab
    # Control -> drug: two different cell types have different source AE encoders.
    for cell_type in ['iPSC','Astrocytes']:
        key,tab=compute_one_ig('control_to_drug',cell_type,drug)
        ig_tables[key]=tab

## 6. Which SOURCE morphology families does IG prioritize?

The source feature universe for each IG run is the respective source AE's retained feature panel. Run the same category representation test (top 10% **mean absolute IG**) against that **source-feature background**, and correct FDR across all IG panels, families and directions. Do not compare IG scores directly across condition-specific models without accounting for their different feature sets and baselines.

In [ ]:
ig_enrich_parts=[]
for label,df in ig_tables.items():
    ig_enrich_parts.append(category_enrichment(df,'IG_mean_abs',label+'__IG_source',q=TOP_QUANTILE))
ig_enrichment=pd.concat(ig_enrich_parts,ignore_index=True)
pvals=np.r_[ig_enrichment.p_enriched.to_numpy(),ig_enrichment.p_depleted.to_numpy()]
qvals=multipletests(pvals,method='fdr_bh')[1]
ig_enrichment['q_enriched']=qvals[:len(ig_enrichment)]
ig_enrichment['q_depleted']=qvals[len(ig_enrichment):]
ig_enrichment['direction_of_representation']=np.where(ig_enrichment.obs_exp>=1,'enriched','depleted')
ig_enrichment['q_direction']=np.where(ig_enrichment.obs_exp>=1,ig_enrichment.q_enriched,ig_enrichment.q_depleted)
ig_enrichment.to_csv(OUT_DIR/'IG_source_family_enrichment.csv',index=False)
for label,table in ig_tables.items():
    analysis=label+'__IG_source'
    enrichment_plot(ig_enrichment,analysis,OUT_DIR/f'{analysis}_enrichment.png')
    top=table.head(20).iloc[::-1]
    fig,ax=plt.subplots(figsize=(8,7))
    ax.barh(top.feature,top.IG_mean_abs,color='#497a94')
    ax.set_xlabel('Mean absolute IG (source standardized morphology features)')
    ax.set_title(label.replace('__',' | '))
    ax.tick_params(axis='y',labelsize=7)
    fig.tight_layout();fig.savefig(OUT_DIR/f'{label}_top20_source_IG.png',dpi=230,bbox_inches='tight');plt.close(fig)
display(ig_enrichment.query('q_direction<0.05').sort_values('q_direction').head(30))

## 7. Interpretation and cautions

- **Measured effect enrichment:** which feature families are over/underrepresented among the largest observed drug effects.
- **Predicted effect enrichment:** whether the model predicts strong effects in those same families. Compare *families* only after respecting the differing feature universes.
- **Signed effect direction:** families with positive versus negative drug effects are described by the signed Cohen's d columns; do not label negative d as category depletion.
- **IG reliance:** which source-family features influence the model's translated **latent response relative to the source training mean baseline**. This is not an attribution of the *drug-vs-DMSO prediction contrast*. For the latter, use an explicitly drug-contrast-specific scalar objective and a defensible cross-condition baseline.
- **Statistical unit:** feature-wise enrichment p-values assume a fixed feature universe; morphology features are strongly correlated, so BH-adjusted p-values are descriptive, not evidence from independent biological replicates. Donor/line-aware validation is still required for claims of population generalization.
- **Two prediction strategies:** the developmental IG source is **drug-treated iPSC morphology**, while control-to-drug IG source is **DMSO morphology** in iPSC or astrocytes. Categories can be compared descriptively, but attribution magnitudes are not directly equivalent.
- **Concordance:** check overlap between top features or family rankings descriptively. Do not interpret source IG feature categories as target effect categories.